# Ollama models

**Benchmark details:**

GPU: *NVIDIA GeForce RTX 4090*

Dataset: *Basic Commands v1.0*

Prompt version: 1 (with longer "json" output required)

In [ ]:
import ollama
import time
import pickle
import pandas as pd
import json
import sys
import os
sys.path.insert(1, '../')

from common import get_test_dataset, get_prompt_template, is_command_classification_correct

X , y = get_test_dataset('../data/basic_commands_v1.0.json')

os.environ["OLLAMA_HOST"] = #AVS host

In [2]:
results = {}

def benchmark_model(model_id: str):
    llm_answers = []
    predictions = {
        'Predicted request': [],
        'Predicted value': [],
        'Classification Result': [],
        'Probability': [],
        'Prompt processing time': []
    }

    probability_threshold = 0.80

    client = ollama.Client()
    
    for i in range(len(X)):

        messages=[
            {"role": "system",  "content": get_prompt_template()},
            {"role": "user",  "content": X[i][0]}]
        
        start_time = time.time() 
        response = client.chat(model=model_id, messages=messages, format='json')    
        end_time = time.time()
        prompt_processing_time = end_time - start_time

        answer = response['message']['content']
        llm_answers.append(answer)
        classification_result = is_command_classification_correct(answer, y[i], probability_threshold)
        predictions['Classification Result'].append(classification_result[0])
        predictions['Prompt processing time'].append(prompt_processing_time)

        if classification_result[0] != "WRONG_OUTPUT_FORMAT":
            predictions['Predicted request'].append(classification_result[1])
            predictions['Predicted value'].append(classification_result[2])
            predictions['Probability'].append(classification_result[3])
        else:
            predictions['Predicted request'].append(None)
            predictions['Predicted value'].append(None)
            predictions['Probability'].append(None)

    predictions = pd.DataFrame(predictions)
    accuracy = predictions[predictions['Classification Result'] == "CLASSIFICATION_CORRECT"].shape[0] / predictions.shape[0] 
    results[model_id] = {"Accuracy" : accuracy, "Predictions" : predictions, "LLM Answers" : llm_answers}

# LLaMA models

### Llama 3.1 8B

In [3]:
benchmark_model("llama3.1:8b") # 100% GPU

### Llama 3.2 3B

In [4]:
benchmark_model("llama3.2:3b") # 100% GPU

# DeepSeek models

### DeepSeek R1 7B

In [5]:
benchmark_model("deepseek-r1:7b") # 100% GPU

### DeepSeek R1 14B

In [6]:
benchmark_model("deepseek-r1:14b") # 100% GPU

### DeepSeek R1 32B

In [7]:
benchmark_model("deepseek-r1:32b") # 7% CPU / 93% GPU

# Gemma models

### Gemma 2 9B

In [8]:
benchmark_model("gemma2:9b") # 100% GPU

### Gemma 2 27B

In [9]:
benchmark_model("gemma2:27b") # 100% GPU

### Gemma 3 12B

In [10]:
benchmark_model("gemma3:12b") # 100% GPU

# Mistral models

### Mistral v0.3 7B

In [11]:
benchmark_model("mistral:7b") # 100% GPU

### Mistral-small 22B

In [12]:
benchmark_model("mistral-small:22b") # 100% GPU

### Mistral-small 24B

In [13]:
benchmark_model("mistral-small:24b") # 100% GPU

# Results

In [14]:
models = ["llama3.1:8b", "llama3.2:3b", "deepseek-r1:7b", "deepseek-r1:14b", "deepseek-r1:32b", "gemma2:9b", "gemma2:27b", "gemma3:12b", "mistral:7b", "mistral-small:22b", "mistral-small:24b"]
print("Models' accuracy")
for model in models:
    print("{}: {:.2f}%".format(model, results[model]["Accuracy"] * 100))

Models' accuracy
llama3.1:8b: 25.88%
llama3.2:3b: 7.06%
deepseek-r1:7b: 5.88%
deepseek-r1:14b: 11.76%
deepseek-r1:32b: 38.82%
gemma2:9b: 40.00%
gemma2:27b: 31.76%
gemma3:12b: 41.18%
mistral:7b: 18.82%
mistral-small:22b: 34.12%
mistral-small:24b: 65.88%


In [15]:
print("Models' average prompt processing time")
for model in models:
    print("{}: {:.2f}s".format(model, results[model]["Predictions"].loc[:, 'Prompt processing time'].mean()))

Models' average prompt processing time
llama3.1:8b: 3.13s
llama3.2:3b: 1.55s
deepseek-r1:7b: 1.64s
deepseek-r1:14b: 2.41s
deepseek-r1:32b: 19.97s
gemma2:9b: 5.24s
gemma2:27b: 10.81s
gemma3:12b: 7.78s
mistral:7b: 3.41s
mistral-small:22b: 8.81s
mistral-small:24b: 8.16s


In [ ]:
with open("all_commands_benchmark_1.pkl", "wb") as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)